# Orbital Debris SQL Queries

This notebook is intentionally SQL-first and contains query work only (no visualizations).

## **Setup And Function Declarations**

In [1]:
import pandas as pd
import sqlite3

def run_query(sql):
    return pd.read_sql(sql, conn)

def query_all_satellites():
    query = """
    SELECT
      norad_id,
      cospar_id,
      object_name,
      launch_year,
      launch_date,
      launch_site,
      owner,
      country_operator,
      object_type
    FROM satellites
    JOIN launch_events ON launch_events.launch_id = satellites.launch_id
    JOIN ownership_operators ON ownership_operators.owner_code = satellites.owner_code;
    """
    return run_query(query)

# Gets a single satellite's information based on NORAD ID
def query_satellite_info(norad_id):
    query = f"""
    SELECT
      norad_id,
      cospar_id,
      object_name,
      launch_year,
      launch_date,
      launch_site,
      owner,
      country_operator,
      object_type
    FROM satellites
    JOIN launch_events ON launch_events.launch_id = satellites.launch_id
    JOIN ownership_operators ON ownership_operators.owner_code = satellites.owner_code
    WHERE norad_id = {norad_id};
    """
    return run_query(query)

def query_satellite_info_list(norad_ids):
    # This will create a comma-separated string of NORAD IDs for the SQL IN clause
    ids = ','.join(str(id) for id in norad_ids)
    
    query = f"""
    SELECT
      norad_id,
      cospar_id,
      object_name,
      launch_year,
      launch_date,
      launch_site,
      owner,
      country_operator,
      object_type
    FROM satellites
    JOIN launch_events ON launch_events.launch_id = satellites.launch_id
    JOIN ownership_operators ON ownership_operators.owner_code = satellites.owner_code
    WHERE norad_id IN ({ids});
    """
    print(f"Querying information for NORAD IDs: {norad_ids}")
    print(f"Executing SQL query:\n{query}")
    return run_query(query)

pd.set_option('display.max_rows', 100)

conn = sqlite3.connect('../data/clean/orbital_debris.db')

In [2]:
df = query_satellite_info(25544)
display(df)

df = query_satellite_info_list([25544, 33591, 43013])
display(df)

df = query_all_satellites()
display(df)

,norad_id,cospar_id,object_name,launch_year,launch_date,launch_site,owner,country_operator,object_type
0,25544,1998-067A,ISS (ZARYA),1998,1998-11-20,Baikonur Cosmodrome,ISS,MULTINATIONAL,PAYLOAD


Querying information for NORAD IDs: [25544, 33591, 43013]
Executing SQL query:

    SELECT
      norad_id,
      cospar_id,
      object_name,
      launch_year,
      launch_date,
      launch_site,
      owner,
      country_operator,
      object_type
    FROM satellites
    JOIN launch_events ON launch_events.launch_id = satellites.launch_id
    JOIN ownership_operators ON ownership_operators.owner_code = satellites.owner_code
    WHERE norad_id IN (25544,33591,43013);
    


,norad_id,cospar_id,object_name,launch_year,launch_date,launch_site,owner,country_operator,object_type
0,25544,1998-067A,ISS (ZARYA),1998,1998-11-20,Baikonur Cosmodrome,ISS,MULTINATIONAL,PAYLOAD
1,33591,2009-005A,NOAA 19,2009,2009-02-06,Vandenberg AFB,US,USA,PAYLOAD
2,43013,2017-073A,NOAA 20 (JPSS-1),2017,2017-11-18,Vandenberg AFB,US,USA,PAYLOAD


,norad_id,cospar_id,object_name,launch_year,launch_date,launch_site,owner,country_operator,object_type
0,5,1958-002B,VANGUARD 1,1958,1958-03-17,AFETR,US,USA,PAYLOAD
1,11,1959-001A,VANGUARD 2,1959,1959-02-17,AFETR,US,USA,PAYLOAD
2,12,1959-001B,VANGUARD R/B,1959,1959-02-17,AFETR,US,USA,ROCKET BODY
3,16,1958-002A,VANGUARD R/B,1958,1958-03-17,AFETR,US,USA,ROCKET BODY
4,20,1959-007A,VANGUARD 3,1959,1959-09-18,AFETR,US,USA,PAYLOAD
...,...,...,...,...,...,...,...,...,...
33229,68201,2026-052D,OBJECT D,2026,2026-03-16,JSC,PRC,CHINA,PAYLOAD
33230,68202,2026-052E,OBJECT E,2026,2026-03-16,JSC,PRC,CHINA,PAYLOAD
33231,68203,2026-052F,OBJECT F,2026,2026-03-16,JSC,PRC,CHINA,PAYLOAD
33232,68204,2026-052G,OBJECT G,2026,2026-03-16,JSC,PRC,CHINA,PAYLOAD


## Primary Question 1: Growth and Decoupling Baseline

In [3]:
# I absolutely detest table aliases in SQL. I find them to be more confusing 
# than helpful, especially when the table names are not that long to begin with.
# I understand that they can be useful in some cases, but I prefer to just write 
# out the full table names for clarity. Nested queries are usually the only time
# I find them to be necessary, and even then I try to avoid them if possible.

q1 = '''
WITH yearly AS (
  SELECT
    launch_events.launch_year AS launch_year,
    COUNT(*) AS objects_launched,
    COUNT(DISTINCT launch_events.launch_id) AS launch_missions,
    SUM(
      CASE
        WHEN UPPER(COALESCE(satellites.object_type, '')) = 'PAYLOAD' THEN 1
        ELSE 0
      END
    ) AS payload_objects
  FROM satellites
  JOIN launch_events ON launch_events.launch_id = satellites.launch_id
  WHERE launch_events.launch_year IS NOT NULL
  GROUP BY launch_events.launch_year
)
SELECT
  launch_year,
  objects_launched,
  launch_missions,
  payload_objects,
  ROUND(100.0 * payload_objects / NULLIF(objects_launched, 0), 2) AS payload_share_pct,
  SUM(objects_launched) OVER (ORDER BY launch_year) AS cumulative_objects
FROM yearly
ORDER BY launch_year;
'''
launch_trend = run_query(q1)
launch_trend.to_csv('../data/clean/queries/pq1_launch_trend.csv', index=False)
launch_trend.head(100)

,launch_year,objects_launched,launch_missions,payload_objects,payload_share_pct,cumulative_objects
0,1958,3,1,1,33.33,3
1,1959,7,5,5,71.43,10
2,1960,13,6,5,38.46,23
3,1961,212,8,9,4.25,235
4,1962,33,15,14,42.42,268
5,1963,91,12,17,18.68,359
6,1964,60,21,28,46.67,419
7,1965,449,33,52,11.58,868
8,1966,207,34,38,18.36,1075
9,1967,95,31,49,51.58,1170


## **Primary Question 2: High-Risk Distribution by Altitude Band**

## **Primary Question 3: Zombie Concentration by Owner**

## **Secondary: Object Type × Operational Status**

## **Secondary: User Category Profile**

## **Extra Questions!**

## **Tidy Up!**

In [4]:
# Close the connection.
# Release resources and ensure clean exit (old habits die hard).
conn.close()
print('Connection closed.')

Connection closed.
